In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import torch
from nanodrz.model import DiarizeGPT, Config
from nanodrz.data import libritts_test, artificial_diarisation_sample
from nanodrz.utils import visualise_annotation, play
from nanodrz.download import dl_scp_file

ckpt = torch.load(dl_scp_file("gpudev:/home/harry/runs/nanodrz/nanodrz/1705840799/0037000.pt"))

In [ ]:
config = Config(**ckpt["config"])
model:DiarizeGPT = DiarizeGPT.from_pretrained(ckpt).cuda()

In [ ]:
print(config.model.use_time_pos)

In [ ]:
# Use the same parameters that the model was trained on to generate a sample
print(config.data.model_dump())
audio, labels = artificial_diarisation_sample(libritts_test(), **config.data.model_dump())
print(audio.shape[-1]/16000)
reference = visualise_annotation(labels)
play(audio)
labels
audio = audio.cuda()

In [ ]:
nlabels = model.generate(audio[None], temperature=.5, max_steps=(len(labels))*3)
print("\n".join([str(n) for n in nlabels]))
for l in nlabels:
    l[2] = l[2]+ "'"
hypothesis = visualise_annotation(labels+nlabels)

### Calculate Metrics with Pyannote


In [ ]:
!pip install pyannote.metrics

In [ ]:
from pyannote.core import Annotation
from pyannote.metrics.diarization import DiarizationErrorRate

metric = DiarizationErrorRate()
metric(reference, hypothesis)